In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path.cwd().parent))

In [2]:
from src.inference_functions import compute_eval_stats
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import random_split, DataLoader
from src.visualization import plot_history
from src.configs import SEED, BATCH_SIZE
from src.train_functions import train
from src.dataset import ImageDataset
from src.loss_function import Loss
from src.model import Model
from pathlib import Path
import torch

In [3]:
from torchvision.transforms import v2
import torch

# The mean and standard deviations across each channel for the normalized pixels
# of every single image in the "trainval" dataset
MEANS = (0.4485, 0.4249, 0.3922)
STDS = (0.2682, 0.2655, 0.2782)

trainval_transforms = v2.Compose([
    v2.Normalize(mean=MEANS, std=STDS)
])

In [4]:
annot_file_trainval = Path("../data/preprocessed/trainval/annotations.csv")
img_dir_trainval = Path("../data/preprocessed/trainval/Images")

trainval_dataset = ImageDataset(annot_file_trainval, img_dir_trainval,
                                transform=trainval_transforms)

generator_ = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(trainval_dataset, [0.05, 0.95]
                                          ,generator=generator_)

len(train_dataset)

251

In [5]:
train_dl = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
num_epochs = 8

In [8]:
model = Model().to(device)
loss_fn = Loss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [9]:
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0

    for X_batch, y_batch in train_dl:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        preds = model(X_batch)
        losses = loss_fn(preds, y_batch)
        loss = losses["average loss"]

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    print(epoch, epoch_loss / len(train_dl))

0 13.234132409095764
1 9.155409395694733
2 7.381397485733032
3 6.2572168707847595
4 5.286930322647095
5 4.436950385570526
6 3.8293462991714478
7 3.3841238915920258


In [10]:
train_mAP = compute_eval_stats(model, train_dl, device)
train_mAP

0.138546568795481